# 07 - CatBoost on Preprocessed Features

Re-tests CatBoost (the strongest model on raw features, Experiment 002)
on the winning feature set from Experiment 005 (income digit
decomposition + frequency encoding), matching the same check already
done for LightGBM in Experiment 007. Loads the saved
`Data/train_preprocessed.csv` / `Data/test_preprocessed.csv` directly
rather than recomputing features inline.

This run was executed directly on the user's machine via
`run_catboost_preprocessed.py` (CatBoost is CPU-only in this
environment and the ~10-15 minute run kept getting interrupted by a
dropped remote connection when attempted here). The code below matches
that script exactly; the results are read back from the saved
`tmp_results/cb_oof_partial.npy` and `tmp_results/cb_test_partial.npy`
arrays and the written submission file, not re-run in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

BASE = ".."

train = pd.read_csv(f"{BASE}/Data/train_preprocessed.csv")
test = pd.read_csv(f"{BASE}/Data/test_preprocessed.csv")
TARGET = "Will_Buy_EV"
y = train[TARGET].map({"No": 0, "Yes": 1})

digit_cols = [f"inc_d{i}" for i in range(6)]
base_cat_cols = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]
cat_cols = base_cat_cols + digit_cols

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))

X = train.drop(columns=[TARGET, "id"]).copy()
Xt = test.drop(columns=["id"]).copy()
for c in cat_cols:
    X[c] = X[c].astype(str)
    Xt[c] = Xt[c].astype(str)

print(f"Feature matrix: {X.shape}, test: {Xt.shape}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

Feature matrix: (668665, 21), test: (286571, 21)
Categorical columns (12): ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'inc_d0', 'inc_d1', 'inc_d2', 'inc_d3', 'inc_d4', 'inc_d5']


## Training (run via `run_catboost_preprocessed.py`, not inline here)

Same CatBoost config as Experiment 002 (`depth=7`, `learning_rate=0.05`,
`iterations=2000`, `early_stopping_rounds=50`), applied to the 21-column
preprocessed feature set instead of the 13 raw columns. The digit
columns are passed as categorical (via `cat_features`), consistent with
how the base categoricals are handled - CatBoost's ordered target
statistics apply to both. Per-fold progress and partial arrays were
saved to `tmp_results/` after every fold specifically so an interrupted
run wouldn't lose completed folds.

In [ ]:
for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=2000, learning_rate=0.05, depth=7,
        loss_function="Logloss", eval_metric="AUC",
        random_seed=42, verbose=False,
    )
    model.fit(X_tr, y_tr, cat_features=cat_cols, eval_set=(X_va, y_va), early_stopping_rounds=50)
    # ... (predict, score, save partial results - see run_catboost_preprocessed.py for the full loop)

## Results (read back from the saved arrays)

Per-fold AUCs recomputed here from the saved OOF predictions against
the same deterministic `StratifiedKFold(random_state=42)` split used
throughout this project - a legitimate way to recover the per-fold
breakdown from a run that happened outside this notebook.

In [ ]:
oof = np.load(f"{BASE}/tmp_results/cb_oof_partial.npy")
test_preds = np.load(f"{BASE}/tmp_results/cb_test_partial.npy")

fold_aucs = [roc_auc_score(y.iloc[va], oof[va]) for _, va in folds]
oof_auc = roc_auc_score(y, oof)

print("Fold AUCs:", [round(a, 6) for a in fold_aucs])
print("Mean fold AUC:", np.mean(fold_aucs), " Std:", np.std(fold_aucs))
print("OOF AUC:", oof_auc)

Fold AUCs: [0.944315, 0.94498, 0.946117, 0.945505, 0.945101]
Mean fold AUC: 0.9452035000654735  Std: 0.0005959543013499567
OOF AUC: 0.9451977581490459


**CatBoost OOF AUC: 0.945198** - higher than XGBoost's preprocessed
score (0.944174, Experiment 005) by +0.001024, and CatBoost's own raw
score (0.941656, Experiment 002) by +0.003542, the largest single-model
gain from this feature set of any of the three families tested.

| Model | Raw OOF | Preprocessed OOF | Δ from preprocessing |
|---|---|---|---|
| XGBoost | 0.941720 | 0.944174 | +0.002454 |
| LightGBM | 0.941470 | 0.943663 | +0.002193 |
| CatBoost | 0.941656 | **0.945198** | **+0.003542** |

**This flips the raw-feature ranking.** On raw features the order was
XGBoost > CatBoost > LightGBM (Experiments 002-004). On the
preprocessed feature set it's now **CatBoost > XGBoost > LightGBM** -
CatBoost's ordered target statistics evidently extract more from the
income-digit categoricals specifically than XGBoost's histogram-based
categorical splits do. This is a genuinely new best model for the
project, not just a confirmation of the existing one.

In [ ]:
test_pred_avg = test_preds.mean(axis=0)

xgb_sub = pd.read_csv(f"{BASE}/submissions/xgboost_5fold_income_digits_freq.csv")
lgb_test = np.load(f"{BASE}/tmp_results/lgb_test.npy")

corr_xgb = np.corrcoef(test_pred_avg, xgb_sub[TARGET].values)[0, 1]
corr_lgb = np.corrcoef(test_pred_avg, lgb_test)[0, 1]

print("CatBoost vs XGBoost correlation:", corr_xgb)
print("CatBoost vs LightGBM correlation:", corr_lgb)

CatBoost vs XGBoost correlation: 0.9957587406616071
CatBoost vs LightGBM correlation: 0.9953295024183648


> [!note] Slightly more diversity than on raw features, still high
> On raw features all three models were >0.998 correlated
> (Experiments 003-004). On the preprocessed feature set that's loosened
> a little - 0.9958 and 0.9953 here, and XGBoost/LightGBM sit at 0.9979
> (checked separately). Not a big change, but it moves in the direction
> that makes stacking (Phase 4) slightly more worthwhile than it looked
> on raw features - the models are still learning very similar
> functions, but marginally less identical ones now.

In [ ]:
submission = pd.DataFrame({"id": test["id"], TARGET: test_pred_avg})

print("Submission shape:", submission.shape)
print("Missing values:", submission.isna().sum().sum())
submission.head()

# Already written by run_catboost_preprocessed.py to:
# submissions/catboost_5fold_income_digits_freq.csv

       id  Will_Buy_EV\n0  668665     0.019961\n1  668666     0.018197\n2  668667     0.003492\n3  668668     0.001188\n4  668669     0.029291

## Experiment 007 (cont.) - CatBoost on Preprocessed Features

- Features: income digit decomposition + frequency encoding (same as
  Experiment 005), loaded from `Data/train_preprocessed.csv` /
  `test_preprocessed.csv`
- CV: identical 5-fold `StratifiedKFold` (`random_state=42`) used
  throughout this project
- Model: CatBoost, same config as Experiment 002 (`depth=7`,
  `learning_rate=0.05`, up to 2000 iterations, early stopping)

**OOF AUC: 0.945198 - new best result for this project**, ahead of
XGBoost's 0.944174 (Experiment 005) by +0.00102.

**Submission ready:** `submissions/catboost_5fold_income_digits_freq.csv`
- not yet sent to Kaggle. Worth submitting: the OOF/LB relationship has
held tightly on every submission so far (5/5), so this is very likely
a genuine leaderboard improvement over the current best of 0.94425/rank
840, not just a CV artifact.

**Next:** submit this to Kaggle. Then Phase 3 (Optuna tuning) and Phase
4 (stacking) should both use CatBoost-on-preprocessed-features as the
new baseline to beat, not XGBoost - and the slightly loosened
correlation between models here makes a stack of all three (on this
feature set) worth trying properly rather than assuming it won't help,
as the raw-feature correlations suggested.